# MedLabel Copilot
### A chatbot that compares exact U.S. over-the-counter product labels

**Research question.** Can semantic matching understand symptom paraphrases, while explicit label rules keep the chatbot grounded when a user changes their age or current medicine?

**Try it.** Upload this notebook to Google Colab and select **Runtime → Run all**. Scroll to **4. Try the chatbot**. The first run may download a 22 MB language model. The catalog is embedded, so a separate CSV upload is optional.

**Scope.** Use fictional scenarios. This project compares selected label fields for seven adult symptoms and 17 exact U.S. formulations. The prototype does not determine diagnosis, personal suitability, dose, timing, or whether combinations are safe. Its emergency phrase check is narrow and does not assess urgency comprehensively.

## 1. Inspect the source dataset

The catalog contains **17 Human OTC Drug Labels** downloaded from the U.S. National Library of Medicine's [DailyMed SPL API](https://dailymed.nlm.nih.gov/dailymed/app-support-web-services.cfm). Each record preserves the product, active ingredients, selected uses and warning flags, DailyMed set ID, version, effective date, and hash of the retrieved XML. A row describes an exact formulation. See the [FDA guide to OTC label sections](https://www.fda.gov/drugs/understanding-over-counter-medicines/over-counter-drug-facts-label) for how these fields are defined.

The catalog covers headache, fever, cough, nasal congestion, runny nose, sore throat, and chest congestion. The chest-congestion tag represents label uses for loosening phlegm and thinning bronchial secretions. Coverage varies by symptom. The warning flags represent selected passages and are incomplete. Ingredient aliases normalize a salt name to its active ingredient for this comparison. For example, dextromethorphan hydrobromide and dextromethorphan are compared as dextromethorphan. The original label names remain in the dataset.

In [ ]:
import csv
from io import StringIO

CATALOG_CSV = 'id,product,label_active_ingredients,normalized_active_ingredients,covered_label_uses,acetaminophen_overlap_warning,nsaid_overlap_warning,age_60_bleeding_warning,liver_disease_warning,liver_cirrhosis_warning,stomach_ulcer_warning,high_blood_pressure_warning,blood_thinner_warning,pregnancy_breastfeeding_warning,maoi_warning,drowsiness_warning,label_set_id,spl_version,spl_effective_date,label_url,spl_api_url,spl_snapshot_sha256\ntylenol_extra,Tylenol Extra Strength tablets,ACETAMINOPHEN,acetaminophen,headache | fever,1,0,0,1,0,0,0,1,1,0,0,103d109d-f520-409c-8da2-eb6b0fbec891,16,2026-07-22,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=103d109d-f520-409c-8da2-eb6b0fbec891&version=16,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/103d109d-f520-409c-8da2-eb6b0fbec891.xml,e04bb147cf015bfcdfaa3f47b5fc80b59a8b679d8f04e5133cd2a30afab9f648\ntylenol_regular,Tylenol Regular Strength tablets,ACETAMINOPHEN,acetaminophen,headache | fever,1,0,0,1,0,0,0,1,1,0,0,01f41fae-4abb-4b96-ab09-a7d3dfc286cd,10,2026-01-26,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=01f41fae-4abb-4b96-ab09-a7d3dfc286cd&version=10,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/01f41fae-4abb-4b96-ab09-a7d3dfc286cd.xml,dc261da23ce22c98f59c2910b2201fb27ff4671182915ecb2b8578b675d31d5a\nadvil,Advil Liqui-Gels,IBUPROFEN,ibuprofen,headache | fever,0,1,1,0,1,1,1,1,1,0,0,3ca6bc9d-090f-e412-e063-6394a90a6876,3,2026-07-03,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=3ca6bc9d-090f-e412-e063-6394a90a6876&version=3,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/3ca6bc9d-090f-e412-e063-6394a90a6876.xml,438ab6980cfd2e319176edfe1b95b9400da434f15778d723fdc4f97245042b2e\nmotrin,Motrin IB caplets,IBUPROFEN,ibuprofen,headache | fever,0,1,1,0,1,1,1,1,1,0,0,52c650cf-3085-7b2c-e063-6394a90a797f,1,2026-05-27,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=52c650cf-3085-7b2c-e063-6394a90a797f&version=1,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/52c650cf-3085-7b2c-e063-6394a90a797f.xml,755842b98b9a7959711147647b2fb7e8a0b3f32a30dba6d73368c70dd13b6456\nbayer,Bayer Genuine Aspirin 325 mg tablets,ASPIRIN,aspirin,headache | fever,0,1,1,0,1,1,1,1,1,0,0,3ca776c2-47ed-5107-e063-6394a90ac5df,5,2026-10-24,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=3ca776c2-47ed-5107-e063-6394a90ac5df&version=5,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/3ca776c2-47ed-5107-e063-6394a90ac5df.xml,6521306ca255c05f65d5285ff77cf55b1d090edb5b27ffcd4050a3e292b03a2d\ndayquil,Vicks DayQuil Cold and Flu liquid,ACETAMINOPHEN | DEXTROMETHORPHAN HYDROBROMIDE | PHENYLEPHRINE HYDROCHLORIDE,acetaminophen | dextromethorphan | phenylephrine,headache | fever | cough | nasal congestion | sore throat,1,0,0,1,0,0,1,1,1,1,0,fc9a43d8-0359-9def-e053-6294a90a7e91,9,2026-10-05,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=fc9a43d8-0359-9def-e053-6294a90a7e91&version=9,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/fc9a43d8-0359-9def-e053-6294a90a7e91.xml,c33dfed1dc16bf5f5f3112c9c1cbea179ea9dd05850af31f1c56f22a335bca96\nnyquil,Vicks NyQuil Cold and Flu liquid,ACETAMINOPHEN | DEXTROMETHORPHAN HYDROBROMIDE | DOXYLAMINE SUCCINATE,acetaminophen | dextromethorphan | doxylamine,headache | fever | cough | runny nose | sore throat,1,0,0,1,0,0,0,1,1,1,1,330d42da-5041-3059-e063-6394a90a2d55,2,2026-10-05,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=330d42da-5041-3059-e063-6394a90a2d55&version=2,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/330d42da-5041-3059-e063-6394a90a2d55.xml,1d11889f663a56359ee76aaab781221179b73cdd37fe7db7e5a01b2e20ff3983\nmucinex,Mucinex DM extended-release tablets,DEXTROMETHORPHAN HYDROBROMIDE | GUAIFENESIN,dextromethorphan | guaifenesin,cough | chest congestion,0,0,0,0,0,0,0,0,1,1,0,70987d06-d206-445f-bd0d-5e1345b8465c,19,2026-06-15,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=70987d06-d206-445f-bd0d-5e1345b8465c&version=19,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/70987d06-d206-445f-bd0d-5e1345b8465c.xml,d55a19f45f5c4618ce138f547689fa55b46d1dc637bc2df5667406694a0ebf2b\ndelsym,Delsym 12 Hour Cough Relief suspension,DEXTROMETHORPHAN,dextromethorphan,cough,0,0,0,0,0,0,0,0,1,1,0,e3e1d126-76c2-4776-8ce0-449625b7e6ba,15,2026-09-09,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=e3e1d126-76c2-4776-8ce0-449625b7e6ba&version=15,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/e3e1d126-76c2-4776-8ce0-449625b7e6ba.xml,776ea08f5a579c7392ca1089502546c9c70dab7056c78f166850756df7005b54\ncvs_dm,CVS Adult Cough Plus Chest Congestion DM solution,DEXTROMETHORPHAN HYDROBROMIDE | GUAIFENESIN,dextromethorphan | guaifenesin,cough | chest congestion,0,0,0,0,0,0,0,0,1,1,0,4490a7e1-d5b9-459c-8751-c24486807981,6,2026-08-19,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=4490a7e1-d5b9-459c-8751-c24486807981&version=6,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/4490a7e1-d5b9-459c-8751-c24486807981.xml,40036d0085b94f1dd06cef1afb0fd5a80cb23c6336b81c8e8b291e5877b0aab7\nmucinex_plain,Mucinex 600 mg guaifenesin extended-release tablets,GUAIFENESIN,guaifenesin,chest congestion,0,0,0,0,0,0,0,0,1,0,0,dd379cdd-90ab-42e0-ad89-f50d3220f611,20,2026-05-27,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=dd379cdd-90ab-42e0-ad89-f50d3220f611&version=20,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/dd379cdd-90ab-42e0-ad89-f50d3220f611.xml,7869a4540bdaf893d76ebbeebe9010163d52a4db792c61207db563b07715e077\nvicks_44,Vicks Formula 44DM Cough liquid,DEXTROMETHORPHAN HYDROBROMIDE,dextromethorphan,cough,0,0,0,0,0,0,0,0,1,1,0,05411a51-a49a-96fd-e063-6294a90ae37b,6,2026-10-05,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=05411a51-a49a-96fd-e063-6294a90ae37b&version=6,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/05411a51-a49a-96fd-e063-6294a90ae37b.xml,b8d80b844b01556465638572cd47055be9a9b3b26826f4d7285312d112845658\nchloraseptic,Chloraseptic Cherry Sore Throat spray,PHENOL,phenol,sore throat,0,0,0,0,0,0,0,0,1,0,0,d5d62f7a-8f92-4d42-a2fb-691f896f72fc,11,2025-09-26,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=d5d62f7a-8f92-4d42-a2fb-691f896f72fc&version=11,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/d5d62f7a-8f92-4d42-a2fb-691f896f72fc.xml,2487463f67f363489c993a6e6306655bacf55b96e61feaef70ddd462a48aecf1\ncepacol,Cepacol Extra Strength Sore Throat Cherry lozenges,"BENZOCAINE | MENTHOL, UNSPECIFIED FORM",benzocaine | menthol,sore throat,0,0,0,0,0,0,0,0,1,0,0,941e36dd-d8d0-4e5c-8585-10e8f06bf8f0,11,2026-05-12,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=941e36dd-d8d0-4e5c-8585-10e8f06bf8f0&version=11,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/941e36dd-d8d0-4e5c-8585-10e8f06bf8f0.xml,33d41fd1686175846a9be3808503f91c5a50bdddaf4e71d3ad5e6314f470a792\nhalls,Halls Cherry menthol lozenges,MENTHOL,menthol,cough | sore throat,0,0,0,0,0,0,0,0,0,0,0,5f4af8e8-8550-45c7-a199-694efac9e7be,18,2025-06-16,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=5f4af8e8-8550-45c7-a199-694efac9e7be&version=18,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/5f4af8e8-8550-45c7-a199-694efac9e7be.xml,baba78ea7637e004203857e4d5378788db32f96f1e739d8ca14c45d0376afadd\nnaproxen,Family Wellness Naproxen Sodium 220 mg tablets,NAPROXEN SODIUM,naproxen,headache | fever,0,1,1,0,1,1,1,1,1,0,0,b6205032-f5c6-494a-8b87-adbbbc3982e7,4,2026-10-14,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=b6205032-f5c6-494a-8b87-adbbbc3982e7&version=4,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/b6205032-f5c6-494a-8b87-adbbbc3982e7.xml,f44147789cbb00c330c931046586b00107bbd26a1609b06c96b229d1a8f5a861\nwalgreens,Walgreens Regular Strength Acetaminophen 325 mg tablets,ACETAMINOPHEN,acetaminophen,headache | fever,1,0,0,1,0,0,0,1,1,0,0,06326fb7-a97b-428a-9cab-8cbdb6074366,4,2022-11-25,https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=06326fb7-a97b-428a-9cab-8cbdb6074366&version=4,https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/06326fb7-a97b-428a-9cab-8cbdb6074366.xml,1e008b0c329ee87b061c3d3b0714ed90b522b1a22997ef5cab79aea4d1170ed8\n'
import pandas as pd

catalog_preview = pd.read_csv(StringIO(CATALOG_CSV))
columns = ["product", "label_active_ingredients", "covered_label_uses", "spl_version", "spl_effective_date"]
display(catalog_preview[columns])


## 2. Match everyday language to a symptom category

Both methods index the same seven short symptom descriptions. TF-IDF compares words and phrases. [MiniLM](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) embeds each sentence as a vector and compares cosine similarity. A small, separate set of synthetic development cases determines each cutoff. The model weights and tokenizer are pinned and verified by checksum. If the download or dependency setup is unavailable, the demo uses TF-IDF.

An exact supported symptom goes straight to confirmation. For other wording, a retrieval score is used only to suggest a category. The user must confirm the category before any product labels are shown. Exact product matching and warning logic are independent of the language model.

In [ ]:
import csv
from dataclasses import dataclass
from hashlib import sha256
from html import escape
from io import StringIO
from pathlib import Path
import re
import subprocess
import sys
from urllib.request import urlopen

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

SYMPTOM_TEXT = {
    "headache": "Headache. Pain or aching in the head or temples.",
    "fever": "Fever. A raised body temperature or feeling feverish.",
    "cough": "Cough. Repeated coughing, a dry cough, or coughing up mucus.",
    "nasal congestion": "Nasal congestion. A blocked or stuffy nose.",
    "runny nose": "Runny nose. Watery discharge or a nose that keeps running.",
    "sore throat": "Sore throat. Minor throat pain, soreness, or a scratchy feeling when swallowing.",
    "chest congestion": "Chest congestion. Thick mucus or phlegm in the chest or bronchial passages.",
}
SYMPTOMS = tuple(SYMPTOM_TEXT)
CORPUS = list(SYMPTOM_TEXT.values())
OUTSIDE = "outside catalog"

# These development cases set the similarity threshold. The later test cases are separate.
CALIBRATION = [
    ("a headache this afternoon", "headache"),
    ("there's an ache in my temples", "headache"),
    ("my body temperature is high", "fever"),
    ("a fever with the cold", "fever"),
    ("I've been coughing today", "cough"),
    ("a hacking dry cough", "cough"),
    ("my nose feels plugged up", "nasal congestion"),
    ("stuffy nose from a cold", "nasal congestion"),
    ("my nose won't stop dripping", "runny nose"),
    ("clear watery discharge from my nose", "runny nose"),
    ("my throat is sore from this cold", "sore throat"),
    ("my throat feels scratchy and hurts a little", "sore throat"),
    ("I have chest congestion with mucus", "chest congestion"),
    ("I feel phlegm sitting in my chest", "chest congestion"),
    ("I sprained my ankle", OUTSIDE),
    ("I have a skin rash", OUTSIDE),
    ("my ears are ringing", OUTSIDE),
    ("my wrist feels stiff", OUTSIDE),
    ("I need help sleeping", OUTSIDE),
    ("I feel sick and don't know why", OUTSIDE),
]

tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
keyword_index = tfidf.fit_transform(CORPUS)


def keyword_scores(texts):
    return (tfidf.transform(texts) @ keyword_index.T).toarray()


MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
MODEL_BASE = f"https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/{MODEL_REVISION}"
MODEL_FILES = {
    "tokenizer.json": "be50c3628f2bf5bb5e3a7f17b1f74611b2561a3a27eeab05e5aa30f411572037",
    "onnx/model_quint8_avx2.onnx": "b941bf19f1f1283680f449fa6a7336bb5600bdcd5f84d10ddc5cd72218a0fd21",
}


def model_file(relative):
    destination = Path(".medlabel_model") / Path(relative).name
    checksum = MODEL_FILES[relative]
    if destination.exists() and sha256(destination.read_bytes()).hexdigest() == checksum:
        return destination
    with urlopen(f"{MODEL_BASE}/{relative}", timeout=45) as response:
        contents = response.read()
    if sha256(contents).hexdigest() != checksum:
        raise ValueError(f"Checksum mismatch for {relative}")
    destination.parent.mkdir(exist_ok=True)
    destination.write_bytes(contents)
    return destination


def load_embeddings():
    try:
        import onnxruntime as ort
        from tokenizers import Tokenizer
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "tokenizers"],
            check=True, timeout=120,
        )
        import onnxruntime as ort
        from tokenizers import Tokenizer

    tokenizer = Tokenizer.from_file(str(model_file("tokenizer.json")))
    tokenizer.enable_truncation(max_length=128)
    tokenizer.enable_padding()
    options = ort.SessionOptions()
    options.intra_op_num_threads = 2
    options.inter_op_num_threads = 1
    session = ort.InferenceSession(
        str(model_file("onnx/model_quint8_avx2.onnx")),
        sess_options=options, providers=["CPUExecutionProvider"],
    )
    inputs = {item.name for item in session.get_inputs()}

    def encode(texts):
        batch = tokenizer.encode_batch(list(texts))
        arrays = {
            "input_ids": np.asarray([item.ids for item in batch], dtype=np.int64),
            "attention_mask": np.asarray([item.attention_mask for item in batch], dtype=np.int64),
            "token_type_ids": np.asarray([item.type_ids for item in batch], dtype=np.int64),
        }
        tokens = session.run(["last_hidden_state"], {k: v for k, v in arrays.items() if k in inputs})[0]
        mask = arrays["attention_mask"][:, :, None]
        pooled = (tokens * mask).sum(axis=1) / mask.sum(axis=1)
        return pooled / np.maximum(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12)

    symptom_vectors = encode(CORPUS)

    def similarity(texts):
        return encode(texts) @ symptom_vectors.T

    return similarity


def predict_from_scores(scores, threshold):
    return [
        SYMPTOMS[score.argmax()] if score.max() >= threshold else OUTSIDE
        for score in scores
    ]


def calibrate(score_fn, grid):
    queries, expected = zip(*CALIBRATION)
    scores = score_fn(queries)
    # In ties, choose the higher threshold to ask for confirmation more often.
    return max(
        grid,
        key=lambda threshold: (
            np.mean(np.array(predict_from_scores(scores, threshold)) == expected), threshold,
        ),
    )


retrieval_methods = {"TF-IDF": (keyword_scores, calibrate(keyword_scores, [0.01, 0.10, 0.20, 0.30, 0.40]))}
try:
    embedding_scores = load_embeddings()
    retrieval_methods["MiniLM"] = (
        embedding_scores, calibrate(embedding_scores, [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]),
    )
except (OSError, ImportError, ValueError, RuntimeError, subprocess.SubprocessError) as error:
    print(f"MiniLM unavailable ({type(error).__name__}). The chatbot will use TF-IDF.")

active_method = "MiniLM" if "MiniLM" in retrieval_methods else "TF-IDF"
print(f"Chat symptom matching: {active_method}")
print("Development thresholds:", {name: cutoff for name, (_, cutoff) in retrieval_methods.items()})


def normalize(text):
    return " ".join(re.findall(r"[a-z0-9]+", text.lower()))


def suggest_symptom(text):
    exact = normalize(text)
    if exact in SYMPTOMS:
        return exact

    score, threshold = retrieval_methods[active_method]
    return predict_from_scores(score([text]), threshold)[0]

## 3. Define the comparison and conversation

The chatbot asks for a confirmed symptom, adult age band, exact current products, and selected health factors. It also accepts supported age, symptom, current-medicine, and health-factor details in a single message. It asks only about missing or unclear details, then shows one draft to confirm. It compares every matching label at once and puts case-specific flags first. The table shows product names, active ingredients, findings, and links to the exact label versions. General review advice appears once per case. Product rows distinguish warnings that match the reported facts from label notes that need clarification. Select one row's product name in the dropdown for a more detailed explanation. Missing or ambiguous current product names lead to a follow-up before the table is shown.

The bot uses templates and cites the labels it used. Its overlap check is distinct from a general drug interaction checker. The same symptom may produce a different result after you change the current medicine or age.

In [ ]:
def split_fields(text):
    return [value.strip() for value in text.split("|") if value.strip()]


catalog = {row["id"]: row for row in csv.DictReader(StringIO(CATALOG_CSV))}
product_lookup = {normalize(row["product"]): key for key, row in catalog.items()}
assert catalog and len(catalog) == len(product_lookup)

INGREDIENTS = {key: frozenset(split_fields(row["normalized_active_ingredients"])) for key, row in catalog.items()}
USES = {key: frozenset(split_fields(row["covered_label_uses"])) for key, row in catalog.items()}
NSAIDS = {"aspirin", "ibuprofen", "naproxen"}

AGE_OPTIONS = ("18–59", "60 or older", "Under 18", "Unsure")
RISK_OPTIONS = (
    "None of these", "Liver disease", "Liver cirrhosis", "Stomach ulcers or bleeding",
    "High blood pressure", "Taking a blood thinner", "Taking an MAOI", "Medication allergy",
    "Pregnant or breastfeeding", "Other or unsure",
)
RISK_RULES = {
    "Liver disease": ("liver_disease_warning", "The label says to ask a doctor before use if you have liver disease."),
    "Liver cirrhosis": ("liver_cirrhosis_warning", "The label says to ask a doctor before use if you have liver cirrhosis."),
    "Stomach ulcers or bleeding": ("stomach_ulcer_warning", "The label warns about stomach bleeding and lists a history of ulcers or bleeding as a risk factor."),
    "High blood pressure": ("high_blood_pressure_warning", "The label says to ask a doctor first if you have high blood pressure."),
    "Taking a blood thinner": ("blood_thinner_warning", "The label includes a warning for people taking a blood thinner."),
    "Taking an MAOI": ("maoi_warning", "The label says to avoid this product during MAOI use and for two weeks after stopping it."),
    "Pregnant or breastfeeding": ("pregnancy_breastfeeding_warning", "The label has a pregnancy and breastfeeding warning. See the exact wording on the label."),
}
CASE_ACTIONS = ("Change current medicines", "Change health factors", "Start over")
CASE_REVIEW = (
    "Ask a pharmacist or clinician to review the reported health factors and exact products before choosing. "
    "Follow any product label instructions to ask a doctor."
)
HELP_URL = "https://medlineplus.gov/ency/article/001927.htm"
URGENT_PHRASE = re.compile(r"\b(?:trouble breathing|difficulty breathing|shortness of breath|can t breathe|cannot breathe|chest pain)\b")


@dataclass(frozen=True)
class LabelComparison:
    key: str
    flags: tuple
    notes: tuple


@dataclass(frozen=True)
class Reply:
    text: str
    choices: tuple = ()
    sources: tuple = ()
    status: str = "question"
    comparison: tuple[LabelComparison, ...] = ()


def product_label(key):
    row = catalog[key]
    return (row["product"], row["label_url"])


def matching_products(symptom):
    return tuple(key for key in catalog if symptom in USES[key])


def resolve_current(text):
    fields = split_fields(text)
    if len(fields) == 1 and normalize(fields[0]) == "none":
        return (), ()
    found, unresolved = [], []
    for field in fields:
        key = product_lookup.get(normalize(field))
        (found if key else unresolved).append(key or field)
    return tuple(dict.fromkeys(found)), tuple(unresolved or (["No answer supplied"] if not fields else []))


INTAKE_EXAMPLE = (
    "I'm 28 and have a headache. I took Tylenol Extra Strength tablets. "
    "None of the listed health factors apply."
)
AGE_MENTION = re.compile(r"\b(?:(?:i m|i am|age|aged) (\d{1,3})|(\d{1,3}) years? old)\b")
MEDICINE_MENTION = re.compile(
    r"\b(?:i (?:already )?(?:took|take|have taken|am taking|m taking)|"
    r"current (?:medicines|medications)(?: are)?)\s+(.+)$"
)
MEDICINE_TIME = re.compile(
    r"\s+(?:earlier today|this morning|this afternoon|last night|yesterday|today|earlier)$"
)
NO_MEDICINES = {
    "no meds", "no medicines", "no medications", "i haven t taken anything",
    "i have not taken anything", "i don t take any medicine", "i don t take any medications",
    "i m not taking any medication", "i m not taking any medicine", "current medicines none",
}
NO_HEALTH_FACTORS = {
    "none of the listed health factors apply", "none of these health factors apply",
    "no listed health factors", "health factors none", "none of these apply",
}
RISK_LOOKUP = {normalize(option): option for option in RISK_OPTIONS}


def age_group(age):
    if age < 18:
        return "Under 18"
    if age <= 59:
        return "18–59"
    return "60 or older" if age <= 120 else None


@dataclass(frozen=True)
class IntakeDraft:
    symptom_text: str
    age: str | None
    medicines: tuple | None
    unresolved: tuple
    risks: tuple
    has_details: bool


def parse_intake(text):
    # Use explicit role phrases so a product someone is considering is not recorded as taken.
    raw_clauses = re.split(r"[.!?;\n]+|\band\s+(?=i\b)", text, flags=re.I)
    ages = {int(first or second) for first, second in AGE_MENTION.findall(normalize(text))}
    age = age_group(next(iter(ages))) if len(ages) == 1 else None
    # A first-person age is not enough to identify the age of a different person.
    if re.search(r"\b(?:my child|my kid|my son|my daughter|my parent|my mom|my dad|my partner)\b", normalize(text)):
        age = None

    symptom_parts = []
    medicine_lists = []
    no_medicines = False
    risks = []
    no_risks = False
    for raw_clause in raw_clauses:
        clause = normalize(raw_clause)
        symptom_clause = clause
        if clause in NO_MEDICINES:
            no_medicines = True
            symptom_clause = ""
        else:
            medicine_clause = normalize(re.sub(r"[,|]", " and ", raw_clause))
            if match := MEDICINE_MENTION.search(medicine_clause):
                symptom_clause = medicine_clause[:match.start()]
                for item in re.split(r"\s+and\s+", match.group(1)):
                    medicine_lists.append(MEDICINE_TIME.sub("", item))

        if clause in NO_HEALTH_FACTORS:
            no_risks = True
            symptom_clause = ""
        else:
            for pattern in (
                r"health factors(?: are)? (.+)",
                r"i (?:have|was diagnosed with) (.+)",
                r"i (?:am|m) (pregnant|breastfeeding)",
                r"i (?:take|am taking|m taking) (a blood thinner|an maoi)",
            ):
                match = re.fullmatch(pattern, clause)
                if not match:
                    continue
                term = match.group(1)
                term = {
                    "pregnant": "pregnant or breastfeeding",
                    "breastfeeding": "pregnant or breastfeeding",
                    "a blood thinner": "taking a blood thinner",
                    "an maoi": "taking an maoi",
                }.get(term, term)
                fields = re.split(r"\s+and\s+", term)
                known = [RISK_LOOKUP[field] for field in fields if field in RISK_LOOKUP]
                if known and len(known) != len(fields):
                    known.append("Other or unsure")
                elif known:
                    symptom_clause = ""
                risks.extend(known)
        symptom_clause = AGE_MENTION.sub("", symptom_clause)
        for product in product_lookup:
            symptom_clause = re.sub(rf"\b{re.escape(product)}\b", "", symptom_clause)
        if symptom_clause.strip():
            symptom_parts.append(symptom_clause.strip())

    found, unresolved = resolve_current(" | ".join(medicine_lists)) if medicine_lists else ((), ())
    # If the message conflicts with itself, leave the field open for clarification.
    if no_medicines and medicine_lists:
        medicines, unresolved = None, ()
    else:
        medicines = found if medicine_lists or no_medicines else None
    positive_risks = tuple(dict.fromkeys(risk for risk in risks if risk != "None of these"))
    if (no_risks or "None of these" in risks) and positive_risks:
        confirmed_risks = ()
    else:
        confirmed_risks = positive_risks or (("None of these",) if no_risks or risks else ())
    return IntakeDraft(" ".join(symptom_parts), age, medicines, unresolved, confirmed_risks, bool(
        ages or medicine_lists or no_medicines or no_risks or risks
    ))


def intake_symptom(text, matcher):
    message = normalize(text)
    if not message:
        return None
    literal = [name for name in SYMPTOMS if re.search(rf"\b{re.escape(name)}\b", message)]
    if len(literal) == 1:
        symptom = literal[0]
        negated = re.search(rf"\b(?:no|without|never had|don t have|do not have) (?:a |any )?{re.escape(symptom)}\b", message)
        return None if negated else symptom
    if len(literal) > 1:
        return None
    guess = matcher(text)
    return guess if guess in SYMPTOMS else None


def label_findings(age, current, risks, candidate):
    row = catalog[candidate]
    ingredients = INGREDIENTS[candidate]
    duplicate = sorted(ingredients & set().union(*(INGREDIENTS[key] for key in current)))
    flags = []
    if duplicate:
        overlaps = [
            f"{catalog[key]['product']} ({', '.join(sorted(ingredients & INGREDIENTS[key]))})"
            for key in current if ingredients & INGREDIENTS[key]
        ]
        flags.append("Shared active ingredient(s): " + ", ".join(duplicate) + ". Overlap with " + ", ".join(overlaps) + ".")
        if "acetaminophen" in duplicate and row["acetaminophen_overlap_warning"] == "1":
            flags.append("The label warns against combining it with another acetaminophen product.")
        else:
            flags.append("Ask a pharmacist before combining these products.")

    if row["nsaid_overlap_warning"] == "1" and any(INGREDIENTS[key] & NSAIDS for key in current):
        flags.append("The reported medicines include another NSAID, a stomach-bleeding risk factor listed on this label.")
    if age == "60 or older" and row["age_60_bleeding_warning"] == "1":
        flags.append("The label lists age 60 or older as a factor that raises the chance of stomach bleeding.")
    relevant_risks = set(risks)
    if "Liver cirrhosis" in relevant_risks:
        relevant_risks.add("Liver disease")
    for risk, (field, message) in RISK_RULES.items():
        if risk in relevant_risks and row[field] == "1":
            flags.append(message)

    notes = []
    if "Liver disease" in risks and "Liver cirrhosis" not in risks and row["liver_cirrhosis_warning"] == "1":
        notes.append(
            "This label says to ask a doctor before use if you have liver cirrhosis. "
            "You reported liver disease. Whether this specific warning applies needs clarification."
        )
    if row["drowsiness_warning"] == "1":
        notes.append("Label warns about drowsiness.")
    status = "shared_ingredient" if duplicate else "label_warning" if flags else "label_match"
    return tuple(flags), tuple(notes), status


def compare_labels(symptom, age, current, risks, candidate):
    if age not in AGE_OPTIONS[:2] or symptom not in USES[candidate]:
        return Reply("The age or symptom falls outside this comparison.", status="outside_scope")

    row = catalog[candidate]
    flags, notes, status = label_findings(age, current, risks, candidate)
    current_summary = ", ".join(catalog[key]["product"] for key in current) or "none reported"
    risk_summary = ", ".join(risks) or "not supplied"
    use_description = (
        f"{row['product']}'s label describes loosening mucus and thinning bronchial secretions."
        if symptom == "chest congestion" else f"{row['product']} lists {symptom} among its uses."
    )
    lines = [
        use_description,
        f"Case details: age {age}. Current medicines: {current_summary}. Selected health factors: {risk_summary}.",
        "Active ingredients: " + ", ".join(sorted(INGREDIENTS[candidate])) + ".",
        *(flags + notes or ("No case-specific flag appeared in the selected checks.",)),
    ]
    if risks != ("None of these",):
        lines.append(CASE_REVIEW)
    lines.append("This is a limited label comparison. Suitability, dose, timing, and other interactions need separate review.")
    return Reply(
        "\n".join(lines),
        choices=("Compare another product", *CASE_ACTIONS),
        sources=tuple(product_label(key) for key in dict.fromkeys([candidate, *current])),
        status=status,
    )


class Chat:
    def __init__(self, matcher=suggest_symptom):
        self.matcher = matcher
        self.reset()

    def reset(self):
        self.stage = "symptom"
        self.symptom = self.age = None
        self.current = self.risks = ()
        self.one_message = False
        self.current_known = False
        self.unresolved_current = ()
        return Reply(
            "Describe your symptom, age, and medicines already taken in one message, "
            "or start with a symptom and answer one question at a time."
        )

    def ask_current(self, text=None):
        self.stage = "current"
        if self.unresolved_current:
            name = self.unresolved_current[0]
            choices = tuple(
                row["product"] for row in catalog.values()
                if f" {normalize(name)} " in f" {normalize(row['product'])} "
            )
            if not choices:
                choices = tuple(row["product"] for row in catalog.values())
            return Reply(
                f'Which exact package did you mean by "{name}"? Choose or type its displayed name. '
                "If it is outside this catalog, bring the package to a pharmacist for comparison.",
                (*choices, "Start over"),
                status="verify_product",
            )
        return Reply(
            text or "Which medicines are already in use? Select the exact package below. For multiple products, type their exact displayed names separated by |.",
            ("None", "I am unsure", *(row["product"] for row in catalog.values())),
        )

    def ask_age(self):
        self.stage = "age"
        return Reply("What is the age for this case? Type it or select a range.", AGE_OPTIONS)

    def continue_intake(self):
        if self.symptom is None:
            self.stage = "confirm"
            return Reply("Which one symptom should I compare labels for?", (*SYMPTOMS, "Something else"), status="confirm_symptom")
        if self.age is None:
            return self.ask_age()
        if self.age not in AGE_OPTIONS[:2]:
            self.stage = "end"
            return Reply("This demo compares adult labels and needs a confirmed adult age range.", ("Start over",), status="outside_scope")
        if not self.current_known or self.unresolved_current:
            return self.ask_current()
        if not self.risks:
            return self.ask_risk()

        self.stage = "intake_review"
        medicines = ", ".join(catalog[key]["product"] for key in self.current) or "None reported"
        return Reply(
            "Check the details I picked up before I compare the labels.\n"
            f"Symptom: {self.symptom} · Age: {self.age}\n"
            f"Medicines already taken: {medicines}\n"
            f"Health factors: {', '.join(self.risks)}",
            ("Confirm details", *CASE_ACTIONS),
            status="confirm_intake",
        )

    def ask_risk(self):
        self.stage = "risk"
        return Reply(
            "Choose the health factors you already know apply, or select Other or unsure. "
            "To list more than one, separate the options with |.",
            RISK_OPTIONS,
        )

    def ask_candidate(self, text=None):
        self.stage = "candidate"
        results = []
        for key in matching_products(self.symptom):
            flags, notes, _ = label_findings(self.age, self.current, self.risks, key)
            results.append(LabelComparison(key, flags, notes))

        # Put case-specific warnings first, then order by name within each group.
        results.sort(key=lambda row: (not bool(row.flags), catalog[row.key]["product"].lower()))
        flagged = sum(bool(row.flags) for row in results)
        current = ", ".join(catalog[key]["product"] for key in self.current) or "none reported"
        selected_risks = ", ".join(self.risks)
        summary = [
            f"Label comparison for {self.symptom} · {len(results)} products · {flagged} with case-specific flags.",
            f"Age {self.age} · Current medicines: {current} · Health factors: {selected_risks}.",
        ]
        if self.risks != ("None of these",):
            summary.append(CASE_REVIEW)
        summary.append(text or "Review the table below or select a product for details. The checks cover only selected label warnings.")
        options = tuple(catalog[row.key]["product"] for row in results)
        return Reply(
            "\n".join(summary),
            (*options, *CASE_ACTIONS),
            sources=tuple(product_label(key) for key in self.current),
            status="comparison",
            comparison=tuple(results),
        )

    def submit(self, text):
        text = text.strip()
        value = normalize(text)
        if value == "start over":
            return self.reset()
        if self.stage in {"candidate", "result", "intake_review"}:
            if value == "change current medicines":
                self.current = ()
                self.current_known = False
                self.unresolved_current = ()
                return self.ask_current()
            if value == "change health factors":
                return self.ask_risk()

        if self.stage == "symptom":
            if not value:
                return Reply("Enter a symptom to continue.")
            if URGENT_PHRASE.search(value):
                self.stage = "end"
                return Reply(
                    "You mentioned a possible emergency symptom. Seek emergency help now. In the U.S., call 911.",
                    ("Start over",), (("MedlinePlus emergency signs", HELP_URL),), "urgent",
                )
            draft = parse_intake(text)
            if draft.has_details:
                self.one_message = True
                self.symptom = intake_symptom(draft.symptom_text, self.matcher)
                self.age = draft.age
                self.current = draft.medicines or ()
                self.current_known = draft.medicines is not None
                self.unresolved_current = draft.unresolved
                self.risks = draft.risks
                return self.continue_intake()
            guess = self.matcher(text)
            self.stage = "confirm"
            prompt = (
                f"Did you mean {guess}? Please confirm using the dropdown or type the category, then press Send."
                if guess != OUTSIDE else "I couldn't match that description. Choose a category if one fits."
            )
            choices = tuple(dict.fromkeys((guess, *SYMPTOMS))) if guess in SYMPTOMS else SYMPTOMS
            return Reply(prompt, (*choices, "Something else"), status="confirm_symptom")

        if self.stage == "confirm":
            if value == "something else":
                self.stage = "end"
                return Reply("This demo covers only the seven listed symptoms.", ("Start over",), status="outside_scope")
            if value not in SYMPTOMS:
                return Reply("Please select one symptom category.", (*SYMPTOMS, "Something else"), status="confirm_symptom")
            self.symptom = value
            return self.continue_intake() if self.one_message else self.ask_age()

        if self.stage == "age":
            age_by_text = {normalize(age): age for age in AGE_OPTIONS}
            self.age = age_group(int(value)) if re.fullmatch(r"[0-9]{1,3}", value) else age_by_text.get(value)
            if self.age is None:
                return self.ask_age()
            if self.age not in AGE_OPTIONS[:2]:
                self.stage = "end"
                return Reply("This demo compares adult labels and needs a confirmed adult age range.", ("Start over",), status="outside_scope")
            return self.continue_intake() if self.one_message else self.ask_current()

        if self.stage == "current":
            found, unresolved = resolve_current(text)
            if self.unresolved_current:
                if unresolved or len(found) != 1:
                    return self.ask_current()
                self.current = tuple(dict.fromkeys((*self.current, *found)))
                self.unresolved_current = self.unresolved_current[1:]
                return self.continue_intake()
            self.current = found
            if unresolved:
                prompt = "I need the exact medicine name and formulation before comparing labels. Choose a listed product, or ask a pharmacist if the package is unclear or outside this catalog."
                reply = self.ask_current(prompt)
                return Reply(reply.text, reply.choices, status="verify_product")
            self.current_known = True
            return self.continue_intake() if self.one_message else self.ask_risk()

        if self.stage == "risk":
            risks = tuple(dict.fromkeys(RISK_LOOKUP.get(normalize(field)) for field in split_fields(text)))
            if not risks or None in risks or ("None of these" in risks and len(risks) > 1):
                return Reply("Please select a listed factor or choose Other or unsure.", RISK_OPTIONS)
            self.risks = risks
            return self.continue_intake() if self.one_message else self.ask_candidate()

        if self.stage == "intake_review":
            if value in {"confirm details", "confirm", "yes"}:
                self.one_message = False
                return self.ask_candidate()
            return self.continue_intake()

        if self.stage == "candidate":
            key = product_lookup.get(value)
            if key not in matching_products(self.symptom):
                return self.ask_candidate("Please select an exact label from the products listed for this symptom.")
            self.stage = "result"
            return compare_labels(self.symptom, self.age, self.current, self.risks, key)

        if self.stage == "result":
            if value == "compare another product":
                return self.ask_candidate()
            return Reply(
                "Choose another product, update the case details, or start over.",
                ("Compare another product", *CASE_ACTIONS),
            )

        return Reply("Start over to begin a new comparison.", ("Start over",))

## 4. Try the chatbot

Click **Try one message** and check the draft, or paste this fictional example into the chat:

> I'm 28 and have a headache. I took Tylenol Extra Strength tablets. None of the listed health factors apply.

Type **Confirm details** and press **Send** to get the table. Try shortening the medicine to **Tylenol** to see an exact-package follow-up. The parser recognizes selected phrasings and listed health factors. If the message leaves out a detail or uses unsupported wording, the chatbot may still ask about it.

Click **Try cough comparison** to open a complete fictional case. The user reports cough and lists **Tylenol Regular Strength tablets** as a current medicine. Inspect how the table distinguishes overlapping formulas from the other labels. Select **Vicks NyQuil Cold and Flu liquid** from the dropdown and press **Send** for details.

Click **Try symptom phrase** to start with a natural-language example. For a manual case, use the dropdown or type one answer per turn, then press **Send**. When the table appears, try **Change current medicines** or **Change health factors** to see it update. The findings reflect a limited set of checks across the catalog.

For the fictional liver-disease example, enter **headache → headache → 18–59 → Tylenol Extra Strength tablets → Liver disease**. Look for a case-level follow-up above the table. Compare the Tylenol row's liver-disease warning with the narrower wording in the Advil row. Choose **Change health factors** to see how explicitly reporting liver cirrhosis changes the output.

In [ ]:
import ipywidgets as widgets
from IPython.display import display


def label_link(title, url):
    return f'<a href="{escape(url, quote=True)}" target="_blank" rel="noopener noreferrer">{escape(title)}</a>'


def comparison_html(comparison):
    if not comparison:
        return ""

    rows = []
    cell = 'style="padding:8px;border-bottom:1px solid #748087;text-align:left;vertical-align:top;overflow-wrap:anywhere"'
    for item in comparison:
        product = label_link(*product_label(item.key))
        ingredients = escape(", ".join(sorted(INGREDIENTS[item.key])))
        findings = []
        if item.flags:
            findings.append("<strong>Review flag</strong><br>" + "<br>".join(map(escape, item.flags)))
        if item.notes:
            findings.append("<strong>Label note</strong><br>" + "<br>".join(map(escape, item.notes)))
        result = "<br>".join(findings) or "No case-specific flag in these checks."
        rows.append(f"<tr><th scope='row' {cell}>{product}</th><td {cell}>{ingredients}</td><td {cell}>{result}</td></tr>")

    headings = "".join(
        f"<th scope='col' {cell}>{name}</th>"
        for name in ("Exact product label", "Active ingredients", "Findings")
    )
    return (
        '<div style="overflow-x:auto;max-width:960px">'
        '<table aria-label="Side-by-side label comparison" style="width:100%;border-collapse:collapse;font-size:inherit">'
        f"<thead><tr>{headings}</tr></thead><tbody>{''.join(rows)}</tbody></table></div>"
        "<p><small>Labels are grouped by flags from selected checks. "
        "A row with no flag still needs a full label review. The table does not assess suitability, dose, timing, or every interaction.</small></p>"
    )


def build_chat_widget():
    chat = Chat()
    transcript = []
    log = widgets.HTML()
    comparison = widgets.HTML()
    field = widgets.Text(placeholder="Type a symptom or answer", description="Message")
    select = widgets.Dropdown(description="Or select", layout=widgets.Layout(width="100%", max_width="600px"))
    send = widgets.Button(description="Send", button_style="primary")
    reset = widgets.Button(description="New case")
    load_example = widgets.Button(description="Try symptom phrase", layout=widgets.Layout(width="175px"))
    load_cough_case = widgets.Button(description="Try cough comparison", layout=widgets.Layout(width="185px"))
    load_intake = widgets.Button(description="Try one message", layout=widgets.Layout(width="165px"))
    error = widgets.HTML()

    def show(reply):
        parts = []
        for speaker, text, sources in transcript[-5:]:
            links = " · ".join(label_link(title, url) for title, url in sources)
            message = escape(text).replace("\n", "<br>")
            parts.append(
                f'<div style="border-left:3px solid #398080;padding:7px 10px;margin:7px 0">'
                f'<strong>{escape(speaker)}</strong><br>{message}'
                + (f"<br><small>{links}</small>" if links else "")
                + "</div>"
            )
        log.value = '<section aria-label="Recent conversation">' + "".join(parts) + "</section>"
        comparison.value = comparison_html(reply.comparison)
        select.options = [("Select an answer", ""), *((choice, choice) for choice in reply.choices)]
        select.value = ""
        select.disabled = not reply.choices
        field.value = ""
        error.value = ""

    def record_turn(message):
        reply = chat.submit(message)
        transcript.extend([("You", message, ()), ("MedLabel", reply.text, reply.sources)])
        return reply

    def send_message(_):
        typed = field.value.strip()
        if typed and select.value:
            error.value = "Use either the text field or the dropdown for this answer."
            return
        message = typed or select.value
        if not message:
            error.value = "Type a message or select an answer."
            return
        show(record_turn(message))

    def restart(_):
        reply = chat.reset()
        transcript[:] = [("MedLabel", reply.text, reply.sources)]
        show(reply)

    def example(_):
        restart(None)
        show(record_turn("A dull ache across my forehead."))

    def cough_case(_):
        restart(None)
        for message in ("cough", "cough", "18–59", catalog["tylenol_regular"]["product"], "None of these"):
            reply = record_turn(message)
        show(reply)

    def intake_example(_):
        restart(None)
        show(record_turn(INTAKE_EXAMPLE))

    send.on_click(send_message)
    reset.on_click(restart)
    load_intake.on_click(intake_example)
    load_example.on_click(example)
    load_cough_case.on_click(cough_case)
    restart(None)
    return widgets.VBox([
        widgets.HTML(
            "<h3>MedLabel Copilot</h3>"
            "<p>Educational U.S. OTC label comparison. Use fictional scenarios. "
            "A result covers only the selected checks in this catalog.</p>"
        ),
        log, field, select,
        widgets.HBox([send, reset, load_intake, load_example, load_cough_case], layout=widgets.Layout(flex_flow="row wrap")),
        error, comparison,
    ])


display(build_chat_widget())

## 5. Evaluate the project

The 38 benchmark utterances below are handcrafted and separate from the 20 cases used for thresholds. A correct result means the proposed category matches the expected answer, or the system declines to suggest a category for an unsupported case. These counts compare the two retrieval methods directly. The chatbot also recognizes exact supported symptom names before calling either method. They describe this benchmark and are not clinical performance estimates.

The scenario tests exercise the conversation and compare controlled changes in age, reported medicines, and selected health factors. They check status, specific reasoning, and source link counts for the covered cases.

In [5]:
import pandas as pd

# Synthetic benchmark cases, separate from the threshold-setting cases.
TEST_QUERIES = [
    ("a mild headache", "headache", "literal"),
    ("my head is pounding", "headache", "paraphrase"),
    ("there's a throbbing feeling behind my forehead", "headache", "paraphrase"),
    ("a dull ache across my forehead", "headache", "paraphrase"),
    ("I have a fever tonight", "fever", "literal"),
    ("the thermometer read higher than usual", "fever", "paraphrase"),
    ("I'm running a temperature", "fever", "paraphrase"),
    ("my temperature went up overnight", "fever", "paraphrase"),
    ("a cough that comes and goes", "cough", "literal"),
    ("I keep having coughing fits", "cough", "paraphrase"),
    ("I can't stop hacking", "cough", "paraphrase"),
    ("I keep hacking up phlegm", "cough", "paraphrase"),
    ("I have nasal congestion", "nasal congestion", "literal"),
    ("my nostrils feel clogged", "nasal congestion", "paraphrase"),
    ("my nose is totally stuffed up", "nasal congestion", "paraphrase"),
    ("it feels like my nose is blocked", "nasal congestion", "paraphrase"),
    ("I have a runny nose", "runny nose", "literal"),
    ("liquid keeps dripping out of my nostrils", "runny nose", "paraphrase"),
    ("my nose keeps leaking like a faucet", "runny nose", "paraphrase"),
    ("my nose keeps running and I need tissues", "runny nose", "paraphrase"),
    ("I have a sore throat today", "sore throat", "literal"),
    ("my throat feels raw", "sore throat", "paraphrase"),
    ("it stings a little when I swallow", "sore throat", "paraphrase"),
    ("a scratch in the back of my throat", "sore throat", "paraphrase"),
    ("my chest congestion is bothering me", "chest congestion", "literal"),
    ("there's mucus that feels stuck in my chest", "chest congestion", "paraphrase"),
    ("my chest feels full of phlegm", "chest congestion", "paraphrase"),
    ("I can't clear all the mucus from my chest", "chest congestion", "paraphrase"),
    ("my knee hurts when I walk", OUTSIDE, "outside"),
    ("I have an upset stomach", OUTSIDE, "outside"),
    ("I have an itchy patch on my arm", OUTSIDE, "outside"),
    ("I have a painful tooth", OUTSIDE, "outside"),
    ("I cut my hand", OUTSIDE, "outside"),
    ("the room feels like it is spinning", OUTSIDE, "outside"),
    ("my eyes are itchy and watery", OUTSIDE, "outside"),
    ("I cannot hear well out of my left ear", OUTSIDE, "outside"),
    ("I want something for heartburn", OUTSIDE, "outside"),
    ("my shoulder aches from lifting boxes", OUTSIDE, "outside"),
]

queries = pd.DataFrame(TEST_QUERIES, columns=["query", "expected", "group"])
metrics = []
for name, (scorer, cutoff) in retrieval_methods.items():
    queries[name] = predict_from_scores(scorer(queries["query"].tolist()), cutoff)
    for group, rows in queries.groupby("group", sort=False):
        metrics.append({
            "method": name, "case group": group,
            "correct": int(rows[name].eq(rows["expected"]).sum()), "cases": len(rows),
        })

print("Held-out synthetic language benchmark")
display(pd.DataFrame(metrics))
display(queries)


def reach_result(symptom, age, current, risk, product):
    chat = Chat()
    chat.submit(symptom)
    chat.submit("headache")
    age_reply = chat.submit(age)
    if age_reply.status == "outside_scope":
        return age_reply
    current_reply = chat.submit(current)
    if current_reply.status == "verify_product":
        return current_reply
    chat.submit(risk)
    return chat.submit(product)


base = ("My head is pounding.", "18–59", "None", "None of these", catalog["tylenol_extra"]["product"])

def vary_case(*, age=base[1], current=base[2], risk=base[3], product=base[4]):
    return (base[0], age, current, risk, product)


checks = [
    ("No medicine reported", base, "label_match", "limited label comparison", 1),
    ("Same headache, DayQuil already used", vary_case(current=catalog["dayquil"]["product"]), "shared_ingredient", "acetaminophen", 2),
    ("Same headache, formulation unknown", vary_case(current="DayQuil"), "verify_product", "exact medicine name", 0),
    ("One exact product and one outside catalog", vary_case(current=catalog["dayquil"]["product"] + " | mystery medicine"), "verify_product", "exact medicine name", 0),
    ("Same product, under 18", vary_case(age="Under 18"), "outside_scope", "adult age range", 0),
    ("Same product, age unknown", vary_case(age="Unsure"), "outside_scope", "adult age range", 0),
    ("Advil at age 18–59", vary_case(product=catalog["advil"]["product"]), "label_match", "limited label comparison", 1),
    ("Advil at age 60+", vary_case(age="60 or older", product=catalog["advil"]["product"]), "label_warning", "stomach bleeding", 1),
    ("Advil with an aspirin product", vary_case(current=catalog["bayer"]["product"], product=catalog["advil"]["product"]), "label_warning", "NSAID", 2),
    ("Tylenol with liver disease reported", vary_case(risk="Liver disease"), "label_warning", "liver disease", 1),
    ("Medication allergy stays a case-level review", vary_case(risk="Medication allergy"), "label_match", CASE_REVIEW, 1),
    ("Advil with unspecified liver disease shows a related label note", vary_case(risk="Liver disease", product=catalog["advil"]["product"]), "label_match", "specific warning applies needs clarification", 1),
    ("Advil with liver cirrhosis reported shows a label flag", vary_case(risk="Liver cirrhosis", product=catalog["advil"]["product"]), "label_warning", "before use if you have liver cirrhosis", 1),
    ("Tylenol with cirrhosis reported shows the broader label warning", vary_case(risk="Liver cirrhosis"), "label_warning", "before use if you have liver disease", 1),
    ("DayQuil with high blood pressure reported", vary_case(risk="High blood pressure", product=catalog["dayquil"]["product"]), "label_warning", "high blood pressure", 1),
    ("Tylenol with pregnancy reported", vary_case(risk="Pregnant or breastfeeding"), "label_warning", "pregnancy and breastfeeding", 1),
    ("Naproxen at age 60+", vary_case(age="60 or older", product=catalog["naproxen"]["product"]), "label_warning", "stomach bleeding", 1),
    ("DayQuil with an MAOI reported", vary_case(risk="Taking an MAOI", product=catalog["dayquil"]["product"]), "label_warning", "two weeks", 1),
]
validation = []
for name, case, expected, evidence, links in checks:
    response = reach_result(*case)
    assert response.status == expected, (name, response)
    assert evidence in response.text and len(response.sources) == links, name
    validation.append({"scenario": name, "expected route": expected, "result": "Pass"})

chat = Chat()
urgent_reply = chat.submit("I am having trouble breathing")
assert urgent_reply.status == "urgent" and urgent_reply.sources
validation.append({"scenario": "Explicit emergency phrase", "expected route": "urgent", "result": "Pass"})

chat = Chat()
chat.submit("my ankle is swollen")
assert chat.submit("Something else").status == "outside_scope"
validation.append({"scenario": "User confirms an unsupported symptom", "expected route": "outside_scope", "result": "Pass"})

for symptom in SYMPTOMS:
    assert suggest_symptom(f" {symptom.upper()}! ") == symptom
    confirmation = Chat().submit(symptom)
    assert f"Did you mean {symptom}?" in confirmation.text
    assert confirmation.choices[0] == symptom
validation.append({"scenario": "Exact names for all seven supported symptoms", "expected route": "confirm_symptom", "result": "Pass"})

chat = Chat()
assert "Did you mean cough?" in chat.submit("cough").text
chat.submit("cough")
chat.submit("18–59")
chat.submit(catalog["delsym"]["product"])
chat.submit("None of these")
response = chat.submit(catalog["mucinex"]["product"])
assert response.status == "shared_ingredient" and "dextromethorphan" in response.text
validation.append({"scenario": "Dextromethorphan across formulations", "expected route": "shared_ingredient", "result": "Pass"})

chat = Chat()
for turn in [*base[:1], "headache", *base[1:]]:
    chat.submit(turn)
chat.submit("Change current medicines")
chat.submit(catalog["dayquil"]["product"])
chat.submit("None of these")
response = chat.submit(catalog["tylenol_extra"]["product"])
assert response.status == "shared_ingredient"
validation.append({"scenario": "Change an answer within the same chat", "expected route": "shared_ingredient", "result": "Pass"})

chat = Chat()
for turn in ("cough", "cough", "18–59", catalog["tylenol_regular"]["product"], "None of these"):
    cough_result = chat.submit(turn)
assert cough_result.status == "comparison"
assert len(cough_result.comparison) == 7
flagged = {row.key for row in cough_result.comparison if row.flags}
assert flagged == {"dayquil", "nyquil"}
assert "acetaminophen" in cough_result.comparison[0].flags[0]
assert "drowsiness" in next(row.notes[0] for row in cough_result.comparison if row.key == "nyquil")
validation.append({"scenario": "Cough with Tylenol: all seven labels, two overlap flags", "expected route": "comparison", "result": "Pass"})

chat.submit("Change current medicines")
assert chat.submit("Tylenol").status == "verify_product"
chat.submit("None")
updated_result = chat.submit("None of these")
assert len(updated_result.comparison) == 7 and not any(row.flags for row in updated_result.comparison)
validation.append({"scenario": "Unclear medicine blocks table and a confirmed change refreshes flags", "expected route": "comparison", "result": "Pass"})

assert "mucinex_plain" in matching_products("chest congestion")
assert "mucinex_plain" not in matching_products("cough")
validation.append({"scenario": "Plain guaifenesin appears for mucus, not general cough", "expected route": "label coverage", "result": "Pass"})

chat = Chat()
for turn in ("headache", "headache", "18–59", catalog["tylenol_extra"]["product"], "Liver disease"):
    liver_result = chat.submit(turn)
assert liver_result.status == "comparison"
assert liver_result.text.count(CASE_REVIEW) == 1
liver_rows = {row.key: row for row in liver_result.comparison}
assert {key for key, row in liver_rows.items() if row.flags} == {
    "tylenol_extra", "tylenol_regular", "dayquil", "nyquil", "walgreens",
}
for key in ("advil", "motrin", "bayer", "naproxen"):
    assert not liver_rows[key].flags
    assert any("liver cirrhosis" in note for note in liver_rows[key].notes)
assert CASE_REVIEW not in comparison_html(liver_result.comparison)
chat.submit("Change health factors")
revised = chat.submit("Liver cirrhosis")
assert len(revised.comparison) == 9 and all(row.flags for row in revised.comparison)
assert sum("Shared active ingredient(s)" in flag for row in revised.comparison for flag in row.flags) == 5
validation.append({"scenario": "Liver case separates label flags, clarification notes, and one case-level review", "expected route": "comparison", "result": "Pass"})

chat = Chat()
draft = chat.submit(INTAKE_EXAMPLE)
assert draft.status == "confirm_intake" and "Tylenol Extra Strength tablets" in draft.text
assert not draft.comparison
assert chat.submit("Confirm details").status == "comparison"
validation.append({"scenario": "Complete first message needs only one confirmation", "expected route": "confirm_intake", "result": "Pass"})

chat = Chat()
draft = chat.submit("I'm 28 and have a headache. I took Advil Liqui-Gels and Tylenol.")
assert draft.status == "verify_product" and "Which exact package" in draft.text
assert set(draft.choices) >= {"Tylenol Extra Strength tablets", "Tylenol Regular Strength tablets"}
risk = chat.submit("Tylenol Regular Strength tablets")
assert "health factors" in risk.text and chat.age == "18–59" and chat.symptom == "headache"
draft = chat.submit("None of these")
assert draft.status == "confirm_intake" and "Advil Liqui-Gels, Tylenol Regular Strength tablets" in draft.text
validation.append({"scenario": "Ambiguous brand asks only for package and retains known medicines", "expected route": "verify_product", "result": "Pass"})

chat = Chat()
draft = chat.submit("I have a headache. I took Tylenol Extra Strength tablets this morning. I have liver disease.")
assert "What is the age" in draft.text
draft = chat.submit("28")
assert draft.status == "confirm_intake" and "Liver disease" in draft.text
validation.append({"scenario": "Missing age asks one question and retains the reported health factor", "expected route": "confirm_intake", "result": "Pass"})

chat = Chat()
draft = chat.submit("I'm 28 and have a headache. No medications. None of the listed health factors apply.")
assert draft.status == "confirm_intake" and "None reported" in draft.text
assert chat.submit("yes").status == "comparison"
validation.append({"scenario": "Explicit no medications and no listed health factors are retained", "expected route": "confirm_intake", "result": "Pass"})

chat = Chat()
draft = chat.submit("I'm 28 and have no headache. I took Advil Liqui-Gels, Tylenol Regular Strength tablets.")
assert draft.status == "confirm_symptom" and chat.symptom is None
risk = chat.submit("cough")
assert "health factors" in risk.text and len(chat.current) == 2
validation.append({"scenario": "Negated symptom asks for clarification and comma-separated medicines stay in the case", "expected route": "confirm_symptom", "result": "Pass"})

chat = Chat()
reply = chat.submit("I'm 16 and have a headache. No medications. None of the listed health factors apply.")
assert reply.status == "outside_scope" and not reply.comparison
validation.append({"scenario": "One-message intake applies the adult scope check", "expected route": "outside_scope", "result": "Pass"})

chat = Chat()
reply = chat.submit("I'm 28. I took Delsym 12 Hour Cough Relief suspension. None of the listed health factors apply.")
assert reply.status == "confirm_symptom" and chat.symptom is None
reply = chat.submit("headache")
assert reply.status == "confirm_intake" and "Symptom: headache" in reply.text
assert "Delsym 12 Hour Cough Relief suspension" in reply.text
validation.append({"scenario": "A medicine name cannot supply a missing symptom", "expected route": "confirm_symptom", "result": "Pass"})

print(f"Rule and dialogue checks: {len(validation)}/{len(validation)} passed.")
display(pd.DataFrame(validation))
print("\nSample side-by-side comparison from the cough demo")
from IPython.display import HTML
display(HTML(comparison_html(cough_result.comparison)))

Held-out synthetic language benchmark
Rule and dialogue checks: 34/34 passed.

Sample side-by-side comparison from the cough demo


,method,case group,correct,cases
0,TF-IDF,literal,7,7
1,TF-IDF,paraphrase,11,21
2,TF-IDF,outside,10,10
3,MiniLM,literal,7,7
4,MiniLM,paraphrase,14,21
5,MiniLM,outside,10,10


,query,expected,group,TF-IDF,MiniLM
0,a mild headache,headache,literal,headache,headache
1,my head is pounding,headache,paraphrase,headache,outside catalog
2,there's a throbbing feeling behind my forehead,headache,paraphrase,outside catalog,headache
3,a dull ache across my forehead,headache,paraphrase,outside catalog,headache
4,I have a fever tonight,fever,literal,fever,fever
5,the thermometer read higher than usual,fever,paraphrase,outside catalog,outside catalog
6,I'm running a temperature,fever,paraphrase,outside catalog,outside catalog
7,my temperature went up overnight,fever,paraphrase,fever,outside catalog
8,a cough that comes and goes,cough,literal,cough,cough
9,I keep having coughing fits,cough,paraphrase,cough,cough


,scenario,expected route,result
0,No medicine reported,label_match,Pass
1,"Same headache, DayQuil already used",shared_ingredient,Pass
2,"Same headache, formulation unknown",verify_product,Pass
3,One exact product and one outside catalog,verify_product,Pass
4,"Same product, under 18",outside_scope,Pass
5,"Same product, age unknown",outside_scope,Pass
6,Advil at age 18–59,label_match,Pass
7,Advil at age 60+,label_warning,Pass
8,Advil with an aspirin product,label_warning,Pass
9,Tylenol with liver disease reported,label_warning,Pass


Exact product label,Active ingredients,Findings
Vicks DayQuil Cold and Flu liquid,"acetaminophen, dextromethorphan, phenylephrine",Review flagShared active ingredient(s): acetaminophen. Overlap with Tylenol Regular Strength tablets (acetaminophen).The label warns against combining it with another acetaminophen product.
Vicks NyQuil Cold and Flu liquid,"acetaminophen, dextromethorphan, doxylamine",Review flagShared active ingredient(s): acetaminophen. Overlap with Tylenol Regular Strength tablets (acetaminophen).The label warns against combining it with another acetaminophen product.Label noteLabel warns about drowsiness.
CVS Adult Cough Plus Chest Congestion DM solution,"dextromethorphan, guaifenesin",No case-specific flag in these checks.
Delsym 12 Hour Cough Relief suspension,dextromethorphan,No case-specific flag in these checks.
Halls Cherry menthol lozenges,menthol,No case-specific flag in these checks.
Mucinex DM extended-release tablets,"dextromethorphan, guaifenesin",No case-specific flag in these checks.
Vicks Formula 44DM Cough liquid,dextromethorphan,No case-specific flag in these checks.


## 6. Explain it in your interview

- **Data:** 17 versioned DailyMed labels, each with a specific formulation and directly traceable ingredient and warning fields.
- **ML experiment:** compare a word-matching baseline with a pretrained sentence encoder. Show one paraphrase both methods handled and one that was difficult.
- **Demo:** open the cough comparison, then change only the medicine already taken. Show the changed flags and open the two exact label references.
- **Limitations:** the corpus is intentionally small, the prompts are synthetic, and the rules cover selected passages. Before public use, this workflow would need broader data and independent review by a qualified pharmacist, plus real-world usability and safety evaluation.